In [ ]:
# current weights → bad predictions → high loss
#                                         ↓
#                               calculate gradients
#                                         ↓
#                               update weights ← optimizer does this
#                                         ↓
#                           better weights → better predictions

In [ ]:
import numpy as np
import nnfs
from nnfs.datasets import spiral_data

nnfs.init()


# ============================================================
# DENSE LAYER — FORWARD + BACKWARD
# ============================================================

class Layer_Dense:
    def __init__(self, n_inputs, n_neurons):
        self.weights = 0.01 * np.random.randn(n_inputs, n_neurons)
        self.biases  = np.zeros((1, n_neurons))

    def forward(self, inputs):
        self.inputs = inputs              # MUST save — needed in backward
        self.output = np.dot(inputs, self.weights) + self.biases

    def backward(self, dvalues):
        # dvalues = gradient flowing in from the next layer (right side)

        # gradient on weights
        # why inputs.T? — chain rule on dot(inputs, weights)
        # shape: (n_inputs, batch) × (batch, n_neurons) = (n_inputs, n_neurons)
        self.dweights = np.dot(self.inputs.T, dvalues)

        # gradient on biases
        # sum across batch — each sample contributed to bias gradient
        # keepdims → stays (1, n_neurons) matching bias shape
        self.dbiases  = np.sum(dvalues, axis=0, keepdims=True)

        # gradient on inputs — passed LEFT to previous layer
        # shape: (batch, n_neurons) × (n_neurons, n_inputs) = (batch, n_inputs)
        self.dinputs  = np.dot(dvalues, self.weights.T)


# ============================================================
# RELU — FORWARD + BACKWARD
# ============================================================

class Activation_ReLU:
    def forward(self, inputs):
        self.inputs = inputs              # save to know which were negative
        self.output = np.maximum(0, inputs)

    def backward(self, dvalues):
        self.dinputs = dvalues.copy()
        # block gradient where input was negative during forward
        # positive inputs → gradient flows through unchanged
        # negative inputs → gradient = 0 (dead neuron)
        self.dinputs[self.inputs <= 0] = 0


# ============================================================
# SOFTMAX — FORWARD
# ============================================================

class Activation_Softmax:
    def forward(self, inputs):
        self.inputs   = inputs
        exp_values    = np.exp(inputs - np.max(inputs, axis=1, keepdims=True))
        probabilities = exp_values / np.sum(exp_values, axis=1, keepdims=True)
        self.output   = probabilities

    def predictions(self, outputs):
        return np.argmax(outputs, axis=1)  # highest prob = predicted class


# ============================================================
# COMBINED SOFTMAX + CROSS ENTROPY
# forward (loss calculation) + backward (simplified gradient)
# ============================================================

class Activation_Softmax_Loss_CategoricalCrossentropy:

    def __init__(self):
        self.activation = Activation_Softmax()

    def forward(self, inputs, y_true):
        # softmax forward
        self.activation.forward(inputs)
        self.output = self.activation.output   # probabilities

        # loss calculation
        samples        = len(inputs)
        y_pred_clipped = np.clip(self.output, 1e-7, 1 - 1e-7)

        if len(y_true.shape) == 1:             # sparse
            correct_confidences = y_pred_clipped[range(samples), y_true]
        elif len(y_true.shape) == 2:           # one-hot
            correct_confidences = np.sum(y_pred_clipped * y_true, axis=1)

        return np.mean(-np.log(correct_confidences))  # scalar loss

    def backward(self, dvalues, y_true):
        samples = len(dvalues)

        # one-hot → sparse
        if len(y_true.shape) == 2:
            y_true = np.argmax(y_true, axis=1)

        self.dinputs = dvalues.copy()

        # THE KEY LINE: subtract 1 from correct class
        # combined gradient of softmax + cross entropy simplifies to this
        self.dinputs[range(samples), y_true] -= 1

        # normalize by batch size
        self.dinputs = self.dinputs / samples

    def calculate(self, output, y):
        sample_losses = self.forward(output, y)
        return sample_losses


# ============================================================
# ADAM OPTIMIZER
# ============================================================

class Optimizer_Adam:
    def __init__(self, learning_rate=0.001, decay=0.0,
                 epsilon=1e-7, beta_1=0.9, beta_2=0.999):
        self.learning_rate         = learning_rate
        self.current_learning_rate = learning_rate
        self.decay                 = decay
        self.iterations            = 0
        self.epsilon               = epsilon
        self.beta_1                = beta_1
        self.beta_2                = beta_2

    def pre_update_params(self):
        if self.decay:
            self.current_learning_rate = self.learning_rate * \
                (1.0 / (1.0 + self.decay * self.iterations))

    def update_params(self, layer):
        if not hasattr(layer, 'weight_cache'):
            layer.weight_momentums = np.zeros_like(layer.weights)
            layer.weight_cache     = np.zeros_like(layer.weights)
            layer.bias_momentums   = np.zeros_like(layer.biases)
            layer.bias_cache       = np.zeros_like(layer.biases)

        # 1st moment — momentum
        layer.weight_momentums = self.beta_1 * layer.weight_momentums + \
                                 (1 - self.beta_1) * layer.dweights
        layer.bias_momentums   = self.beta_1 * layer.bias_momentums   + \
                                 (1 - self.beta_1) * layer.dbiases

        # bias correction — momentum
        weight_momentums_corrected = layer.weight_momentums / \
            (1 - self.beta_1 ** (self.iterations + 1))
        bias_momentums_corrected   = layer.bias_momentums   / \
            (1 - self.beta_1 ** (self.iterations + 1))

        # 2nd moment — rmsprop
        layer.weight_cache = self.beta_2 * layer.weight_cache + \
                             (1 - self.beta_2) * layer.dweights ** 2
        layer.bias_cache   = self.beta_2 * layer.bias_cache   + \
                             (1 - self.beta_2) * layer.dbiases  ** 2

        # bias correction — cache
        weight_cache_corrected = layer.weight_cache / \
            (1 - self.beta_2 ** (self.iterations + 1))
        bias_cache_corrected   = layer.bias_cache   / \
            (1 - self.beta_2 ** (self.iterations + 1))

        # final weight update
        layer.weights += -self.current_learning_rate * \
                          weight_momentums_corrected / \
                         (np.sqrt(weight_cache_corrected) + self.epsilon)
        layer.biases  += -self.current_learning_rate * \
                          bias_momentums_corrected   / \
                         (np.sqrt(bias_cache_corrected)   + self.epsilon)

    def post_update_params(self):
        self.iterations += 1


# ============================================================
# FULL TRAINING LOOP — FORWARD + BACKWARD + UPDATE
# ============================================================

X, y = spiral_data(samples=100, classes=3)

# build network
dense1         = Layer_Dense(2, 64)
relu1          = Activation_ReLU()
dense2         = Layer_Dense(64, 3)
loss_activation = Activation_Softmax_Loss_CategoricalCrossentropy()
optimizer      = Optimizer_Adam(learning_rate=0.05, decay=5e-7)

for epoch in range(10001):

    # ── FORWARD ──────────────────────────────────────────
    dense1.forward(X)                        # input → hidden
    relu1.forward(dense1.output)             # non-linearity
    dense2.forward(relu1.output)             # hidden → output

    # softmax + loss in one shot
    loss = loss_activation.forward(dense2.output, y)

    # accuracy
    predictions = np.argmax(loss_activation.output, axis=1)
    accuracy    = np.mean(predictions == y)

    if not epoch % 1000:
        print(f'epoch: {epoch:5d} | '
              f'loss: {loss:.4f} | '
              f'acc: {accuracy:.4f} | '
              f'lr: {optimizer.current_learning_rate:.6f}')

    # ── BACKWARD ─────────────────────────────────────────
    # start from loss+softmax → go left layer by layer
    loss_activation.backward(loss_activation.output, y)  # dvalues for dense2
    dense2.backward(loss_activation.dinputs)              # dvalues for relu1
    relu1.backward(dense2.dinputs)                        # dvalues for dense1
    dense1.backward(relu1.dinputs)                        # dvalues stop here

    # ── UPDATE ───────────────────────────────────────────
    optimizer.pre_update_params()        # decay lr
    optimizer.update_params(dense1)      # adjust dense1 weights+biases
    optimizer.update_params(dense2)      # adjust dense2 weights+biases
    optimizer.post_update_params()       # increment iteration

In [ ]:
# Gradient flow 
# FORWARD:
# X(300,2) → dense1 → (300,64) → relu1 → (300,64) → dense2 → (300,3) → softmax+loss → scalar

# BACKWARD:
# scalar → dinputs(300,3) → dense2.dinputs(300,64) → relu1.dinputs(300,64) → dense1.dinputs(300,2)
#               ↓                    ↓
#          dweights(64,3)      dweights(2,64)
#          dbiases(1,3)        dbiases(1,64)
#               ↓                    ↓
#          update dense2        update dense1